# 04 — Respuestas del estudio y cierre de resultados

El cierre integra la comparación principal de regresores y la aplicación secundaria de búsqueda. Las lecturas siguientes se generan con las tablas del informe seleccionado: no fijan un ganador distinto con prueba ni repiten cálculos científicos.

La contribución es un procedimiento reproducible que relaciona utilidad multiobjetivo observada, error, estabilidad, costo e interpretabilidad, declarando dónde la evidencia histórica deja de representar una evaluación física.

In [1]:
%matplotlib inline
%config InlineBackend.figure_formats = {"png", "svg"}
import pandas as pd
from IPython.display import Markdown, display

from truss_research.evaluacion.presentacion import notebook_context

context = notebook_context()
CONFIG = context.config
STUDY_DIR, REPORT_DIR = context.study_dir, context.report_dir
DATA_DIR = CONFIG.data_dir
display(
    Markdown(
        f"**Estudio:** `{STUDY_DIR.name}` — **etiqueta:** `{CONFIG.label}` — **informe:** `{REPORT_DIR.name}`"
    )
)
if CONFIG.label == "comprobacion":
    display(
        Markdown(
            "**Comprobación técnica: sus métricas no constituyen resultados de tesis.**"
        )
    )

**Estudio:** `comparacion_tres_objetivos_final_002` — **etiqueta:** `posterior` — **informe:** `informe_001`

## 1. Decisiones congeladas

`freeze.json` fija la selección, modelos, políticas y validación antes de consultar prueba. `evolution/closed.json` acredita terminación. Una familia diferente puede lograr menor error en prueba sin cambiar la elección de validación. Iniciar prueba no equivale a completar el cierre; el exportador exige cierre completo.

In [2]:
from truss_research.datos.artefactos import read_artifact

closed = read_artifact(STUDY_DIR / "evolution" / "closed.json")
display(
    pd.Series(
        {
            "fase": closed["phase"],
            "selección_congelada": context.frozen["selected"],
            "firma_del_estudio": context.frozen["signature"],
            "prueba_usada_para_entrenar": context.frozen[
                "test_labels_used_for_training"
            ],
        }
    )
)

fase                                                                   complete
selección_congelada                             [xgboost, random_forest, ridge]
firma_del_estudio             7ff5b56b40c77deeae749211bd746666f4d98202976dc1...
prueba_usada_para_entrenar                                                False
dtype: object

## 2. Respuesta a la pregunta principal y aplicación secundaria

La pregunta principal se responde comparando utilidad observada, error y tiempo. La familia elegida por AUC de validación mantiene su rol, aunque otras familias lideren métricas de prueba. Esos cambios de orden muestran por qué precisión y utilidad no son criterios intercambiables; se describen sin reabrir la selección.

Para la pregunta secundaria, se informa qué estrategias presentan mayores promedios y qué contrastes delimitan la comparación de RL con sus controles. A recupera diseños con respuestas históricas; B optimiza respuestas predichas. Una ventaja en B no valida diseños nuevos ni garantiza una ventaja histórica.

In [3]:
from truss_research.evaluacion.lectura_resultados import (
    predictor_interpretation,
    search_interpretation,
)

display(pd.read_csv(REPORT_DIR / "predictor_utility_summary.csv"))
display(pd.read_csv(REPORT_DIR / "predictive_model_summary.csv"))
display(pd.read_csv(REPORT_DIR / "comparison.csv"))


display(
    Markdown(
        predictor_interpretation(
            REPORT_DIR, context.frozen["selected"][0], min(CONFIG.budgets)
        )
    )
)
display(Markdown(search_interpretation(STUDY_DIR, REPORT_DIR, max(CONFIG.budgets))))

,split,model,role,budget,mean_auc,mean_hypervolume,mean_normalized_auc,effective_budget,catalogue_size
0,validation,catboost,regressor,32,0.064709,0.081835,0.716293,32,96
1,validation,catboost,regressor,64,0.074075,0.084204,0.819961,64,96
2,validation,catboost,regressor,96,0.077500,0.084397,0.857871,96,96
3,validation,dummy,regressor,32,0.062671,0.077687,0.693732,32,96
4,validation,dummy,regressor,64,0.071474,0.082316,0.791168,64,96
...,...,...,...,...,...,...,...,...,...
73,test,targetwise,regressor,64,0.075148,0.084585,0.831845,64,96
74,test,targetwise,regressor,96,0.078324,0.084746,0.866997,96,96
75,test,xgboost,regressor,32,0.065913,0.081517,0.729615,32,96
76,test,xgboost,regressor,64,0.074521,0.083808,0.824903,64,96


,split,model,mean,std,min,max,adjustment_seeds
0,test,catboost,0.065732,0.001112,0.064079,0.067179,5
1,test,dummy,0.780447,0.000000,0.780447,0.780447,5
2,test,gpr,0.079224,0.000000,0.079224,0.079224,5
3,test,lightgbm,0.076244,0.000000,0.076244,0.076244,5
4,test,mlp,0.074468,0.007250,0.067611,0.085514,5
5,test,random_forest,0.080116,0.000528,0.079228,0.080591,5
6,test,ridge,0.236341,0.000000,0.236341,0.236341,5
7,test,svr,0.074392,0.000000,0.074392,0.074392,5
8,test,tabpfn,0.050223,0.000573,0.049247,0.050742,5
9,test,targetwise,0.050223,0.000573,0.049247,0.050742,5


,split,experiment,strategy,budget,mean_auc,mean_hypervolume,attainment_rate,mean_seconds,mean_evaluations,mean_generated,...,mean_duplicates,mean_unsupported_queries,mean_projection_distance,mean_cheap_predictions,mean_generation_seconds,mean_selection_seconds,mean_evaluation_seconds,mean_cheap_seconds,mean_design_distance,mean_topology_count
0,test,A,contextual_bandit,32,0.686238,0.070953,0.00,0.251641,32.0,768.0,...,18.51,0.60,0.038521,0.00,0.123649,0.072657,0.002025,0.000000,0.325475,2.65
1,test,A,contextual_bandit,64,0.741986,0.072941,0.00,0.598055,64.0,1792.0,...,53.80,0.72,0.035181,0.00,0.295970,0.175739,0.004704,0.000000,0.313318,2.65
2,test,A,contextual_bandit,96,0.767118,0.074396,0.01,0.948053,96.0,2816.0,...,90.93,1.27,0.034157,0.00,0.468746,0.277699,0.007381,0.000000,0.305462,2.65
3,test,A,contextual_bandit_assisted,32,0.685762,0.071135,0.00,0.312785,32.0,768.0,...,18.53,0.45,0.037651,730.68,0.123896,0.066735,0.001977,0.065506,0.321173,2.65
4,test,A,contextual_bandit_assisted,64,0.743833,0.073976,0.00,0.747083,64.0,1792.0,...,51.43,0.99,0.035294,1711.78,0.296040,0.163926,0.004680,0.156697,0.311580,2.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,validation,B,rl,64,0.653795,0.064539,0.00,1.461243,64.0,1792.0,...,5.30,0.00,0.000000,0.00,0.291139,0.199858,0.829718,0.000000,0.245817,3.39
104,validation,B,rl,96,0.678210,0.066465,0.00,2.344275,96.0,2816.0,...,7.84,0.00,0.000000,0.00,0.464501,0.357831,1.300707,0.000000,0.248417,3.60
105,validation,B,rl_assisted,32,0.610926,0.062217,0.00,0.682904,32.0,768.0,...,2.19,0.00,0.000000,709.14,0.125394,0.070390,0.356347,0.067526,0.316232,3.67
106,validation,B,rl_assisted,64,0.656287,0.065055,0.00,1.614512,64.0,1792.0,...,4.59,0.00,0.000000,1665.62,0.302001,0.185639,0.826808,0.155717,0.287738,3.90


Con presupuesto **32**, la mayor media descriptiva de AUC normalizado en prueba corresponde a **lightgbm (0.7457)**. La familia elegida en validación, **xgboost**, alcanza **0.7296** y conserva su rol congelado: el orden de prueba no se usa para cambiarla. El menor MAE normalizado medio corresponde a **tabpfn, targetwise (0.0502)**. Estos criterios responden preguntas distintas: reconstruir respuestas con menor error no garantiza seleccionar mejores alternativas multiobjetivo. Las posiciones son descriptivas; los contrastes pareados delimitan la incertidumbre y no establecen superioridad poblacional.

Con presupuesto **96**, los mayores promedios descriptivos de AUC de prueba son:
- **A, recuperación histórica: de (0.7836)**.
- **B, entorno predictivo: de_assisted (0.7708)**.

Los contrastes congelados permiten precisar esta lectura:
- Modo A: rl − contextual_bandit: diferencia media **-0.0254**, intervalo exploratorio **[-0.0456, -0.0033]**, que queda completamente por debajo de cero.
- Modo A: rl − random: diferencia media **-0.0101**, intervalo exploratorio **[-0.0302, +0.0090]**, que incluye cero.
- Modo B: rl − contextual_bandit: diferencia media **-0.0507**, intervalo exploratorio **[-0.0775, -0.0257]**, que queda completamente por debajo de cero.
- Modo B: rl − random: diferencia media **-0.0247**, intervalo exploratorio **[-0.0439, -0.0049]**, que queda completamente por debajo de cero.

Una posición media menor no prueba equivalencia. Estos resultados describen las implementaciones y presupuestos evaluados; no descartan otros algoritmos RL o ajustes. Las consultas sin topología histórica, las proyecciones y los duplicados también condicionan el uso del presupuesto. B no aporta validación estructural independiente.

## 3. Viabilidad computacional e interpretación de las predicciones

La elección práctica depende del uso previsto: entrenamiento ocasional, consultas individuales o evaluación por lotes. Se comparan fases de ajuste registradas y latencias medidas; no se atribuye ahorro al solver original ni se equipara costo local con el preentrenamiento externo de una familia.

SHAP caracteriza qué entradas explican las predicciones y dónde coinciden las familias. No determina el mejor predictor, causalidad física ni seguridad. La estabilidad observada entre semillas comparte datos, partición e hiperparámetros; no equivale a réplicas estructurales independientes. La ausencia de varias semillas SHAP no puede suplirse con el acuerdo entre familias.

In [4]:
from truss_research.evaluacion.lectura_resultados import (
    shap_interpretation,
    time_comparison,
    time_interpretation,
)

display(pd.read_csv(REPORT_DIR / "category_coverage.csv"))
display(pd.read_csv(REPORT_DIR / "predictor_costs.csv"))
display(pd.read_csv(REPORT_DIR / "policy_costs.csv"))


display(time_comparison(REPORT_DIR).round(3))
display(Markdown(time_interpretation(REPORT_DIR, context.frozen["selected"][0])))
display(Markdown(shap_interpretation(REPORT_DIR, CONFIG.representative_seed)))

,web_config,unique_designs,split,scarce_evidence,scarcity_threshold,interpretation
0,0,10,train,False,10,descriptive_threshold_not_statistical_power
1,1,393,train,False,10,descriptive_threshold_not_statistical_power
2,2,441,train,False,10,descriptive_threshold_not_statistical_power
3,3,46,train,False,10,descriptive_threshold_not_statistical_power
4,4,11,train,False,10,descriptive_threshold_not_statistical_power
5,0,2,validation,True,10,descriptive_threshold_not_statistical_power
6,1,39,validation,False,10,descriptive_threshold_not_statistical_power
7,2,57,validation,False,10,descriptive_threshold_not_statistical_power
8,3,0,validation,True,10,descriptive_threshold_not_statistical_power
9,4,0,validation,True,10,descriptive_threshold_not_statistical_power


,model,seed,normalized_mae,cv_normalized_mae,tuning_seconds,crossfit_seconds,fit_seconds,reuse_seconds,device,requested_trials,trial_count,nested_trial_count,targetwise_family_trials,search_reused
0,dummy,42,0.786876,0.785196,0.000955,0.701126,0.006398,0.000000,cpu,30,0,0,0,False
1,dummy,43,0.786876,0.785196,0.000000,0.000000,0.010375,0.014907,cpu,30,0,0,0,True
2,dummy,44,0.786876,0.785196,0.000000,0.000000,0.006590,0.037336,cpu,30,0,0,0,True
3,dummy,45,0.786876,0.785196,0.000000,0.000000,0.008531,0.018040,cpu,30,0,0,0,True
4,dummy,46,0.786876,0.785196,0.000000,0.000000,0.008458,0.015342,cpu,30,0,0,0,True
5,ridge,42,0.242709,0.251220,13.638864,65.523292,0.036498,0.000000,cpu,30,30,150,0,False
6,ridge,43,0.242709,0.251220,0.000000,0.000000,0.024757,0.019432,cpu,30,0,0,0,True
7,ridge,44,0.242709,0.251220,0.000000,0.000000,0.016990,0.014771,cpu,30,0,0,0,True
8,ridge,45,0.242709,0.251220,0.000000,0.000000,0.013391,0.018838,cpu,30,0,0,0,True
9,ridge,46,0.242709,0.251220,0.000000,0.000000,0.023141,0.026604,cpu,30,0,0,0,True


,policy,seed,learning_objective,training_generated,training_cheap_evaluations,training_member_predictions,training_cheap_member_predictions,validation_generated,validation_cheap_evaluations,validation_member_predictions,validation_cheap_member_predictions,wall_seconds,cpu_seconds,training_surrogate_evaluations,optimizer_updates,validation_surrogate_evaluations
0,contextual_bandit,42,immediate,1600000,0,150000,0,1075200,0,33600,0,4733.645101,4385.015625,57248,12251,38400
1,contextual_bandit,43,immediate,1600032,0,150000,0,1075200,0,33600,0,4749.333727,4397.234375,57272,12251,38400
2,contextual_bandit,44,immediate,1600032,0,150000,0,1075200,0,33600,0,3487.463255,3417.703125,57128,12251,38400
3,contextual_bandit,45,immediate,1600032,0,150000,0,1075200,0,33600,0,3506.031008,3423.593750,57144,12251,38400
4,contextual_bandit,46,immediate,1600032,0,150000,0,1075200,0,33600,0,3477.038227,3404.968750,57360,12251,38400
5,contextual_bandit_assisted,42,immediate,1600000,1521073,150000,1521073,1075200,1020410,33600,1020410,3839.737378,3766.062500,57248,12251,38400
6,contextual_bandit_assisted,43,immediate,1600032,1518707,150000,1518707,1075200,1014636,33600,1014636,3833.049859,3759.203125,57272,12251,38400
7,contextual_bandit_assisted,44,immediate,1600032,1509806,150000,1509806,1075200,1009034,33600,1009034,3833.831135,3752.906250,57128,12251,38400
8,contextual_bandit_assisted,45,immediate,1600032,1521676,150000,1521676,1075200,1021437,33600,1021437,3818.091822,3746.968750,57144,12251,38400
9,contextual_bandit_assisted,46,immediate,1600032,1520714,150000,1520714,1075200,1022291,33600,1022291,3558.427438,3507.093750,57360,12251,38400


,tuning_seconds,crossfit_seconds,fit_seconds,fit_phase_minutes,adjustment_seeds,device,individual_ms,batch_ms,batch_per_design_ms,batch_rows
model,,,,,,,,,,
catboost,1100.756,5208.190,59.482,106.140,5,cpu,2.597,2.920,0.027,108.0
dummy,0.001,0.701,0.040,0.012,5,cpu,1.323,1.296,0.012,108.0
gpr,255.539,581.905,52.818,14.838,5,cpu,2.317,5.392,0.050,108.0
lightgbm,31.896,156.576,0.843,3.155,5,cpu,3.374,5.710,0.053,108.0
mlp,168.227,528.148,15.571,11.866,5,cpu,1.859,1.890,0.018,108.0
random_forest,97.703,599.218,5.211,11.702,5,cpu,14.522,17.114,0.158,108.0
ridge,13.639,65.523,0.115,1.321,5,cpu,2.029,2.047,0.019,108.0
svr,40.252,131.927,4.170,2.939,5,cpu,2.621,15.954,0.148,108.0
tabpfn,0.001,30.075,10.154,0.671,5,cuda,407.797,431.532,3.996,108.0


Las fases registradas de ajuste de **xgboost** suman **4.48 min**. La mayor suma corresponde a **catboost (106.14 min)**. La inferencia individual de **xgboost** registra una media entre semillas de medianas de **3.000 ms**. Las familias con menor MAE normalizado, **tabpfn, targetwise**, registran **413.690 ms** como promedio de esas latencias. La tabla y el gráfico permiten valorar si la precisión adicional compensa la latencia para el uso previsto. La suma de fases de ajuste corresponde al trabajo registrado de todas las semillas y preparaciones compartidas; no es el tiempo transcurrido de campaña ni incluye el preentrenamiento externo de TabPFN. Las fases OOF incluyen su selección interna y no deben sumarse de nuevo a los totales de recursos. CPU y CUDA se indican porque sus tiempos no constituyen una comparación a hardware idéntico.

En la semilla representativa **42**, excluyendo el modelo constante:
- **mass_kg**: `n_trusses` ocupa el primer puesto en **10/10** familias, con **64.7–75.9%** de su importancia absoluta total.
- **util_ratio**: `n_trusses` ocupa el primer puesto en **10/10** familias, con **66.7–79.6%** de su importancia absoluta total.
- **z_disp_m**: `n_trusses` ocupa el primer puesto en **10/10** familias, con **38.6–52.6%** de su importancia absoluta total.

La concordancia describe modelos sobre una misma fuente, no réplicas independientes ni causalidad física. Los porcentajes normalizan la media de |SHAP| dentro de cada modelo y respuesta; no representan varianza explicada. Las familias con componentes idénticos pueden compartir atribuciones. Sin varias semillas explicadas no se estima estabilidad SHAP entre ajustes.

## 4. Límites físicos y trabajo pendiente

A36, fluencia de 250 MPa, Eurocódigo 3, geometría y FEM proceden de declaraciones humanas registradas. La ficha no sustituye su evidencia original. Siguen pendientes cargas, apoyos, conectividad, catálogo y asignación de secciones, solver, dimensionamiento y agregación exacta de respuestas.

La utilización promedio no verifica cada barra; L/700 es un criterio operativo adoptado. Las configuraciones nuevas no tienen certificación estructural y no existe medición comparable del evaluador original para afirmar ahorro FEM. La estabilidad entre semillas no equivale a estabilidad cinemática.

La fuente contiene una sola trayectoria y el conjunto de prueba ya había sido examinado. El cierre conserva carácter exploratorio. Las topologías ausentes de prueba no tienen evaluación final en ese bloque; la interpolación, si se ejecuta, es una sensibilidad separada. Ningún promedio de utilización certifica cada barra.

Estas limitaciones acotan la conclusión, pero no eliminan la utilidad de comparar los métodos bajo condiciones comunes. Las mejoras de RL, nuevos problemas estructurales y una validación FEM independiente quedan como trabajo futuro, sin convertirlos en requisitos pendientes de esta campaña cerrada.

In [5]:
display(pd.read_csv(REPORT_DIR / "physical_provenance.csv"))
display(Markdown((REPORT_DIR / "RESULTS.md").read_text(encoding="utf-8")))

,element,status,evidence,confirmation_required,confirmation_source,confirmed_at
0,Archivo y hoja,Confirmado,"240214-C1-20m.xlsx, TOTAL DATA; manifiesto rep...",Conservar identidad de la fuente.,Registro histórico de respuesta de la profesor...,2026-10-19
1,SHA-256,Confirmado,f05c11f9a980497dcf5b74f3b74547244ce020552f95dc...,Verificar al preparar.,Registro histórico de respuesta de la profesor...,2026-10-19
2,Acceso y autorización,Informado por el autor,Fuente autorizada de acceso restringido; LICENSE.,Conservar evidencia de autorización sin public...,NaN,NaN
3,"Cerchas de acero 2D, luz 20 m",Confirmado,Luz fija de 20 m; alturas y secciones variables.,No interpretar luz fija como sección fija.,Registro histórico de respuesta de la profesor...,2026-10-19
4,Origen en optimización topológica,Informado por el autor,Optimización de alternativas geométricas y sel...,"Precisar algoritmo, software, versiones y pará...",NaN,NaN
5,Análisis estructural FEM,Confirmado,La profesora confirma análisis mediante elemen...,"Precisar solver, software y versión; FEM no id...",Registro histórico de respuesta de la profesor...,2026-10-19
6,Cargas y apoyos,Pendiente,No constan en el Excel ni en la respuesta.,"Cargas, combinaciones, apoyos, separación y ví...",NaN,NaN
7,Materiales y secciones,Confirmado,Acero A36 con fluencia 250 MPa; comprobaciones...,250 MPa no es una tensión admisible universal.,Registro histórico de respuesta de la profesor...,2026-10-21
8,Asignación y propiedades por barra,Pendiente,El optimizador seleccionaba secciones por barr...,"Catálogo completo, perfiles por barra, conecti...",NaN,NaN
9,Generaciones y soluciones,Confirmado,"100 generaciones sucesivas (0 a 99), con 50 ap...",No interpretar la solución 0 como diseño base ...,Registro histórico de respuesta de la profesor...,2026-10-19


# Comparación de predictores y propuesta de búsqueda guiada

El resultado principal es la utilidad multiobjetivo de cada regresor: predictor_utility_summary.csv y predictor_utility_paired_intervals.csv. La calidad HV/AUC usa respuestas históricas observadas tras seleccionar; todos comparten catálogo, inicialización, ranking y presupuesto. La familia se elige por AUC de validación en el menor presupuesto configurado, con MAE como desempate. El oráculo aplica el mismo ranking con etiquetas: no es un óptimo global ni una cota de AUC. Los presupuestos efectivos se reducen si el catálogo se agota, sin consultas ficticias. La precisión y costo complementan esta utilidad: predictive_model_summary.csv, predictive_metric_summary.csv, predictive_seed_metrics.csv y predictive_paired_errors.csv. Diferencias negativas favorecen al modelo frente a la referencia. Los contrastes principales usan Dummy y Ridge; los restantes son secundarios. Los intervalos del 95 % son puntuales, exploratorios y condicionados a los ajustes; no son pruebas de significación ni de equivalencia. generation_block remuestrea bloques completos; design_iid_sensitivity es un control. Pocos bloques limitan la estabilidad. Las semillas no multiplican el tamaño muestral.

category_coverage.csv identifica categorías sin evaluación y physical_provenance.csv documenta condiciones conocidas y pendientes. El MAE normalizado usa desviaciones de entrenamiento y pesos iguales; no es un porcentaje de error. Los errores físicos y costos fundamentan la interpretación; no hay tolerancias de equivalencia inventadas.

La búsqueda es una aplicación secundaria exploratoria. A usa respuestas históricas mediante proyección; B usa estimaciones de un sustituto. No se acredita factibilidad física de configuraciones nuevas ni ahorro FEM.

Las consultas históricas sin topología compatible consumen presupuesto y no terminan el episodio. Su frecuencia se informa junto con distancias de proyección. B usa las semillas de la familia elegida; el desacuerdo puede ser cero en modelos deterministas y no constituye incertidumbre física calibrada.

predictive_inference_costs.csv separa latencia individual y por lote. predictor_costs.csv registra ajustes anidados y dispositivo efectivo; resource_tasks.csv consolida intentos, CPU y memoria, y research_cost.csv distingue duración del coordinador predictivo de suma de duraciones de trabajos. Treinta intentos no garantizan igual tiempo de cómputo entre familias. TabPFN incorpora preentrenamiento externo cuyo costo no se midió; Targetwise incluye selección adicional por respuesta.

Las diferencias e intervalos pareados están en evolution/test/paired_intervals.csv. Las consultas no alcanzadas siguen censuradas en summary.csv.

El umbral de alcance usa exclusivamente la referencia de entrenamiento. Los intervalos son exploratorios, sin corrección por comparaciones múltiples.

El tiempo de entrenamiento de políticas se registra aparte del tiempo por búsqueda. No sumar fases anidadas del entrenamiento predictivo.

SHAP es un análisis complementario de validación. Si existe una etapa completa compatible, se adjuntan shap_global.csv, shap_examples.csv, shap_costs.csv y shap_stability.csv. Su ausencia se resuelve con make explain y una nueva exportación; no se calcula al presentar. Las atribuciones describen predicciones, no efectos causales ni seguridad estructural.


## 5. Estado de la campaña y comunicación de resultados

`evolution/closed.json`, la auditoría y los manifiestos de exportación documentan el cierre computacional. Las cuatro fuentes canónicas se conservan sin salidas; `make notebooks ESTUDIO=...` lee tablas y genera una nueva presentación con notebooks ejecutados, HTML y figuras PNG/SVG inventariadas. No entrena, no predice y no calcula SHAP.

La entrega debe conservar la versión del código, configuración, entorno, procedencia y artefactos autorizados. No se sobrescriben presentaciones firmadas. Para abrir las fuentes se usa `TRUSS_STUDY` y, opcionalmente, `TRUSS_REPORT`; para abrir la exportación se ejecuta `make dev ESTUDIO=...`.

Con resultados y cobertura revisados, resta su integración en la memoria y defensa de tesis, incluyendo alcances de costos, incertidumbre y procedencia pendiente. Las campañas y exportaciones pesadas continúan siendo manuales según AGENTS.md.